In [2]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error


In [3]:
df_jan_2023 = pd.read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet')
df_feb_2023 = pd.read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet')

In [4]:
df_jan_2023.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,1.0,N,161,141,2,9.3,1.00,0.5,0.00,0.0,1.0,14.30,2.5,0.00
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,1.0,N,43,237,1,7.9,1.00,0.5,4.00,0.0,1.0,16.90,2.5,0.00
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,1.0,N,48,238,1,14.9,1.00,0.5,15.00,0.0,1.0,34.90,2.5,0.00
3,1,2023-01-01 00:03:48,2023-01-01 00:13:25,0.0,1.90,1.0,N,138,7,1,12.1,7.25,0.5,0.00,0.0,1.0,20.85,0.0,1.25
4,2,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,1.0,N,107,79,1,11.4,1.00,0.5,3.28,0.0,1.0,19.68,2.5,0.00


### Q1. Downloading the data

In [5]:
df_jan_2023.shape

(3066766, 19)

### Q2. Computing duration

In [6]:
df_jan_2023['duration'] = (df_jan_2023['tpep_dropoff_datetime'] - df_jan_2023['tpep_pickup_datetime']).dt.total_seconds() / 60

In [7]:
df_jan_2023['duration'].std()

42.59435124195458

### Q3. Dropping outliers

In [8]:
outliers_removed = df_jan_2023[df_jan_2023['duration'].between(1,60)]

In [9]:
len(outliers_removed)/len(df_jan_2023)

0.9812202822125979

### Q4. One-hot encoding

In [10]:
outliers_removed.loc[:,'PULocationID'] = outliers_removed['PULocationID'].astype(str)
outliers_removed.loc[:,'DOLocationID'] = outliers_removed['DOLocationID'].astype(str)

df_selected = outliers_removed[['PULocationID', 'DOLocationID']]
list_of_dicts = df_selected.to_dict(orient='records')

/tmp/ipykernel_17366/846576287.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['161' '43' '48' ... '114' '230' '262']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  outliers_removed.loc[:,'PULocationID'] = outliers_removed['PULocationID'].astype(str)
/tmp/ipykernel_17366/846576287.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['141' '237' '238' ... '239' '79' '143']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  outliers_removed.loc[:,'DOLocationID'] = outliers_removed['DOLocationID'].astype(str)


In [11]:
# Fit a dictionary vectorizer
vectorizer = DictVectorizer(sparse=True)
feature_matrix = vectorizer.fit_transform(list_of_dicts)

# Feature names
feature_names = vectorizer.get_feature_names_out()

In [12]:
feature_matrix.shape

(3009173, 515)

In [13]:
X = feature_matrix
y = outliers_removed['duration']

### Q5. Training a model

In [14]:
model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)

In [15]:
mse = mean_squared_error(y, y_pred)
rmse = np.sqrt(mse)

In [16]:
rmse

7.6492624397080675

### Q6. Evaluating the model

In [17]:
df_feb_2023['duration'] = (df_feb_2023['tpep_dropoff_datetime'] - df_feb_2023['tpep_pickup_datetime']).dt.total_seconds() / 60

In [18]:
feb_outliers_removed = df_feb_2023[df_feb_2023['duration'].between(1,60)]

In [21]:
feb_outliers_removed.loc[:,'PULocationID'] = feb_outliers_removed['PULocationID'].astype(str)
feb_outliers_removed.loc[:,'DOLocationID'] = feb_outliers_removed['DOLocationID'].astype(str)
feb_selected = feb_outliers_removed[['PULocationID', 'DOLocationID']]

In [25]:
list_of_dicts_feb = feb_selected.to_dict(orient='records')

X_val = vectorizer.transform(list_of_dicts_feb)
target = feb_outliers_removed['duration']

In [26]:
y_pred_val = model.predict(X_val)

In [27]:
mse = mean_squared_error(target, y_pred_val)
rmse = np.sqrt(mse)

In [28]:
rmse

7.81181211389241